In [1]:
# environment

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Optional, useful for local CUDA stability
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM GB:", round(props.total_memory / 1024**3, 1))

CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM GB: 95.0


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

In [4]:
# dataset loading

def load_my_dataset(train_file, validation_file=None):
    data_files = {"train": train_file}

    if validation_file is not None:
        data_files["validation"] = validation_file

    dataset = load_dataset("json", data_files=data_files)

    print("Train examples:", len(dataset["train"]))
    if "validation" in dataset:
        print("Validation examples:", len(dataset["validation"]))

    print("\nDataset columns:", dataset["train"].column_names)
    print("\nFirst example keys:", dataset["train"][0].keys())

    return dataset

In [5]:
#helper to inspect one formatted example

def inspect_chat_template(tokenizer, dataset, max_chars=3000):
    example = dataset["train"][0]["messages"]

    text = tokenizer.apply_chat_template(
        example,
        tokenize=False,
        add_generation_prompt=False,
    )

    print(text[:max_chars])
    print("\nTotal characters:", len(text))

In [7]:
#  training function

def finetune_medgemma_local(
    dataset,
    model_name_or_path="google/medgemma-27b-text-it",
    output_dir="models/medgemma-27b-report-generation-lora",
    batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=1e-4,
    max_seq_length=8192,
    resume_from_checkpoint=None,
    assistant_only_loss=True,
    debug_max_train_samples=None,
    debug_max_eval_samples=None,
    use_flash_attention_2=False,
):
    """
    single-GPU LoRA SFT training.

    Expected dataset format:
    {
      "messages": [
        {"role": "user", "content": "..."},
        {"role": "assistant", "content": "..."}
      ]
    }

    Important:
    - Do NOT pre-render messages into a 'text' field when using assistant_only_loss=True.
    - Keep the structured messages column.
    """

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. This setup expects a local GPU.")

    print("Loading tokenizer from:", model_name_or_path)

    tokenizer = AutoTokenizer.from_pretrained(
        model_name_or_path,
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"


    GEMMA_ASSISTANT_ONLY_CHAT_TEMPLATE = """{{ bos_token }}
    {%- if messages[0]['role'] == 'system' -%}
        {%- set first_user_prefix = messages[0]['content'] + '\\n\\n' -%}
        {%- set loop_messages = messages[1:] -%}
    {%- else -%}
        {%- set first_user_prefix = "" -%}
        {%- set loop_messages = messages -%}
    {%- endif -%}
    
    {%- for message in loop_messages -%}
        {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) -%}
            {{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}
        {%- endif -%}
    
        {%- if message['role'] == 'user' -%}
            {{ '<start_of_turn>user\\n' + first_user_prefix + (message['content'] | trim) + '<end_of_turn>\\n' }}
        {%- elif message['role'] == 'assistant' -%}
            {{ '<start_of_turn>model\\n' }}
            {%- generation -%}
                {{ (message['content'] | trim) + '<end_of_turn>\\n' }}
            {%- endgeneration -%}
        {%- else -%}
            {{ raise_exception('Only user and assistant roles are supported, except optional initial system message.') }}
        {%- endif -%}
    {%- endfor -%}
    
    {%- if add_generation_prompt -%}
        {{ '<start_of_turn>model\\n' }}
    {%- endif -%}
    """
    
    if assistant_only_loss:
        print("Patching tokenizer.chat_template for assistant_only_loss=True")
        tokenizer.chat_template = GEMMA_ASSISTANT_ONLY_CHAT_TEMPLATE
        print("Has generation tag:", "{% generation" in tokenizer.chat_template)    

    print("Loading model from:", model_name_or_path)

    model_kwargs = dict(
        torch_dtype=torch.bfloat16,
        device_map={"": 0},   # use the 96gb
        trust_remote_code=True,
    )

    if use_flash_attention_2:
        model_kwargs["attn_implementation"] = "flash_attention_2"

    model = AutoModelForCausalLM.from_pretrained(
        model_name_or_path,
        **model_kwargs,
    )

    model.config.use_cache = False

    # Gradient checkpointing reduces activation memory.
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

    # Helpful with PEFT + gradient checkpointing.
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    train_dataset = dataset["train"]
    eval_dataset = dataset["validation"] if "validation" in dataset else None

    # Optional tiny debug subset for notebook testing.
    if debug_max_train_samples is not None:
        train_dataset = train_dataset.select(
            range(min(debug_max_train_samples, len(train_dataset)))
        )

    if eval_dataset is not None and debug_max_eval_samples is not None:
        eval_dataset = eval_dataset.select(
            range(min(debug_max_eval_samples, len(eval_dataset)))
        )

    print("\nFormatted example preview:\n")
    preview = tokenizer.apply_chat_template(
        train_dataset[0]["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    print(preview[:3000])

    training_args = SFTConfig(
        output_dir=output_dir,

        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate,
        warmup_ratio=0.05,

        logging_steps=10,

        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,

        eval_strategy="steps" if eval_dataset is not None else "no",
        eval_steps=50 if eval_dataset is not None else None,

        bf16=True,
        fp16=False,

        optim="adamw_torch",

        max_length=max_seq_length,
        packing=False,

        # train only on assistant/report tokens, not on the user instruction.
        assistant_only_loss=assistant_only_loss,

        metric_for_best_model="eval_loss" if eval_dataset is not None else None,
        greater_is_better=False if eval_dataset is not None else None,
        load_best_model_at_end=True if eval_dataset is not None else False,

        report_to=[],
    )

    callbacks = []
    if eval_dataset is not None:
        callbacks.append(
            EarlyStoppingCallback(early_stopping_patience=5)
        )
    if assistant_only_loss:
        print("\nChecking assistant token mask before trainer creation...")
    
        test = tokenizer.apply_chat_template(
            train_dataset[0]["messages"],
            tokenize=True,
            return_dict=True,
            return_assistant_tokens_mask=True,
            add_generation_prompt=False,
        )
    
        mask = test.get("assistant_masks", None)
    
        if mask is None:
            raise RuntimeError("No assistant_masks returned by tokenizer.")
    
        assistant_token_count = int(sum(mask))
        total_token_count = len(test["input_ids"])
    
        print("Assistant tokens:", assistant_token_count)
        print("Total tokens:", total_token_count)
    
        if assistant_token_count == 0:
            raise RuntimeError("Assistant mask is all zeros.")
    
        print("Assistant-only mask OK.")

    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args,
        callbacks=callbacks,
    )

    # Sanity check for assistant-only mask.
    if assistant_only_loss:
        print("\nChecking assistant token mask...")

        test = trainer.processing_class.apply_chat_template(
            train_dataset[0]["messages"],
            tokenize=True,
            return_dict=True,
            return_assistant_tokens_mask=True,
            add_generation_prompt=False,
        )

        mask = (
            test.get("assistant_masks", None)
            or test.get("assistant_tokens_mask", None)
        )

        if mask is None:
            raise RuntimeError(
                "assistant_only_loss=True is set, but no assistant token mask was produced. "
                "MedGemma's chat template may not have been patched with generation markers."
            )

        assistant_token_count = int(sum(mask))
        total_token_count = len(test["input_ids"])

        print("Assistant tokens:", assistant_token_count)
        print("Total tokens:", total_token_count)

        if assistant_token_count == 0:
            raise RuntimeError(
                "Assistant token mask exists but contains 0 assistant tokens. "
                "Do not train until the chat template issue is fixed."
            )

        print("Assistant-only loss mask OK.")

    print("\nStarting training...")
    trainer.train(resume_from_checkpoint=resume_from_checkpoint)

    os.makedirs(output_dir, exist_ok=True)

    print("\nSaving LoRA adapter to:", output_dir)
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    print("Fine-tuning finished.")

    return trainer, tokenizer, model

In [1]:
# load model: 
# from huggingface_hub import login

# login()

In [5]:
# from huggingface_hub import snapshot_download

# local_model_dir = snapshot_download(
#     repo_id="google/medgemma-27b-text-it",
#     local_dir="HuggingFace_Models/google/medgemma-27b-text-it",
#     local_dir_use_symlinks=False,
#     token=True,
# )

# print("Model downloaded to:", local_model_dir)

/opt/conda/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

Model downloaded to: /home/jovyan/work/Aime26/EMNLP/FineTuneGeneration/HuggingFace_Models/google/medgemma-27b-text-it


In [9]:
dataset = load_my_dataset(
    train_file="llm_dataset_output/train.jsonl",
    validation_file="llm_dataset_output/validation.jsonl",
)

trainer, tokenizer, model = finetune_medgemma_local(
    dataset=dataset,
    model_name_or_path="HuggingFace_Models/google/medgemma-27b-text-it",
    output_dir="models/medgemma-27b-report-generation-lora",

    batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=1e-4,
    max_seq_length=8192,

    assistant_only_loss=True,
)

Train examples: 1216
Validation examples: 136

Dataset columns: ['messages']

First example keys: dict_keys(['messages'])
Loading tokenizer from: HuggingFace_Models/google/medgemma-27b-text-it


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Patching tokenizer.chat_template for assistant_only_loss=True
Has generation tag: False
Loading model from: HuggingFace_Models/google/medgemma-27b-text-it


Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Truncating eval dataset:   0%|          | 0/136 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.



Checking assistant token mask...
Assistant tokens: 365
Total tokens: 1457
Assistant-only loss mask OK.

Starting training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.141586,0.145848,0.623845,606070.000000,0.952950
100,0.113714,0.118791,0.644425,1211729.000000,0.959432
150,0.097206,0.104450,0.643400,1819091.000000,0.963482
200,0.076032,0.097611,0.626648,2427243.000000,0.966181
250,0.073322,0.091214,0.619374,3033324.000000,0.968390
300,0.066824,0.086407,0.619780,3639218.000000,0.969997
304,0.066824,0.086149,0.619760,3687242.000000,0.970010



Saving LoRA adapter to: models/medgemma-27b-report-generation-lora
Fine-tuning finished.


In [ ]:
# 89gb